# Загрузка modes из API satnogs

без синхронизированного справочника не заработает отложенная задача, запускаемая по запросу к нашему FastAPI
`GET /api/v2/satellite/update-satellite/`

этот блокнот запрашивает из API режимы модуляции с их id,
потом обновляет существующие и создаёт недостающие в нашей БД

In [1]:

# настройка PYTHONPATH
import os
import sys
from pathlib import Path
from pprint import pformat

sys.path.insert(0, str(Path('../').resolve()))
print(pformat(sys.path))
print(f'{os.getcwd()=}')


['/home/user/proj/soniks',
 '/usr/lib/python312.zip',
 '/usr/lib/python3.12',
 '/usr/lib/python3.12/lib-dynload',
 '',
 '/home/user/proj/soniks/.venv/lib/python3.12/site-packages']
os.getcwd()='/home/user/proj/soniks/_ju'


In [2]:
# подгрузка settgins env из .env

from dotenv import load_dotenv
import os

# Загружаем переменные из .env файла
local_dot_env = str(Path('.local.env').resolve())
print(f'{local_dot_env=}')

load_dotenv('../.local.env')

# Проверяем что переменные загрузились

print("POSTGRES__HOST:", os.getenv("POSTGRES__HOST"))
print("POSTGRES__DB:", os.getenv("POSTGRES__DB"))

local_dot_env='/home/user/proj/soniks/_ju/.local.env'
POSTGRES__HOST: localhost
POSTGRES__DB: soniks


In [4]:

import asyncio
import httpx
from dishka import AsyncContainer, make_async_container
from sqlalchemy.ext.asyncio import AsyncSession, async_sessionmaker

from src.core.configs import settings
from src.core.configs.admin import AdminSettings
from src.core.configs.auth import AuthSettings
from src.core.configs.database import PostgresSettings, SQLEngineSettings
from src.core.configs.file import FileSettings
from src.core.containers import get_providers

from src.infrastructure.postgres.models import ModeORM


URL_BASE_SATNOGS_API = "https://db.satnogs.org/api"
URL_PATH_SATNOGS_MODES = "/modes"

TIMEOUT_SATNOGS_API = 10

async def fetch(url, sem):
    async with sem:  # ограничение на число одновременных задач
        async with httpx.AsyncClient() as client:
            rs = await client.get(url, timeout=TIMEOUT_SATNOGS_API)
            rs.raise_for_status()
            return rs.json()


async def _satnogs_modes():
    sem = asyncio.Semaphore(10)  # максимум 10 конкурентных запросов
    # urls = ["https://httpbin.org/get"] * 1000
    # tasks = [fetch(u, sem) for u in urls]
    # results = await asyncio.gather(*tasks)
    # print(len(results))
    modes_url = f"{URL_BASE_SATNOGS_API}{URL_PATH_SATNOGS_MODES}/"
    # modes_url = f"http://localhost:8088/api/v2/satellite/modes/"  # FIXME: [debug] remove before commit

    raw_modes = await fetch(modes_url, sem)
    print(pformat(raw_modes))
    return raw_modes


def _dishka_app_container() -> AsyncContainer:
    # инициализируем окружение единообразно с основным FastAPI-приложением
    dishka_context = {
        PostgresSettings: settings.postgres,
        SQLEngineSettings: settings.sql_engine,
        AuthSettings: settings.auth,
        AdminSettings: settings.admin,
        FileSettings: settings.file,
    }
    # enter APP scope
    return make_async_container(*get_providers(), context=dishka_context)


async def load_modes():
    app_container_factory = _dishka_app_container()
    
    # активация request-scope контейнера dishka
    async with app_container_factory() as request_container:
        session = await request_container.get(AsyncSession)

        await session.commit()
        for raw_mode in await _satnogs_modes():
            await session.merge(
                ModeORM(
                    id=raw_mode['id'],
                    name=raw_mode['name'],
                )
            )
        await session.commit()
        print('[DONE]')


await load_modes()


[{'id': 90, 'name': '4FSK'},
 {'id': 49, 'name': 'AFSK'},
 {'id': 78, 'name': 'AFSK TUBiX10'},
 {'id': 17, 'name': 'AHRPT'},
 {'id': 19, 'name': 'AM'},
 {'id': 44, 'name': 'APT'},
 {'id': 93, 'name': 'ASK'},
 {'id': 50, 'name': 'BPSK'},
 {'id': 83, 'name': 'BPSK PMT-A3'},
 {'id': 59, 'name': 'CERTO'},
 {'id': 6, 'name': 'CW'},
 {'id': 101, 'name': 'DATV'},
 {'id': 95, 'name': 'DBPSK'},
 {'id': 96, 'name': 'DOKA'},
 {'id': 97, 'name': 'DPSK'},
 {'id': 71, 'name': 'DQPSK'},
 {'id': 98, 'name': 'DSB'},
 {'id': 57, 'name': 'DSTAR'},
 {'id': 58, 'name': 'DUV'},
 {'id': 92, 'name': 'DVB-S2'},
 {'id': 82, 'name': 'FFSK'},
 {'id': 1, 'name': 'FM'},
 {'id': 7, 'name': 'FMN'},
 {'id': 72, 'name': 'FSK'},
 {'id': 84, 'name': 'FSK AX.100 Mode 5'},
 {'id': 85, 'name': 'FSK AX.100 Mode 6'},
 {'id': 88, 'name': 'FSK AX.25 G3RUH'},
 {'id': 100, 'name': 'FT8'},
 {'id': 75, 'name': 'GFSK'},
 {'id': 68, 'name': 'GFSK Rktr'},
 {'id': 94, 'name': 'GFSK/BPSK'},
 {'id': 63, 'name': 'GMSK'},
 {'id': 91, 'name